In [1]:
# conda vis2
from pathlib import Path
import os
import sys
import tempfile
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import re

PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

# Use a project-local temp directory so Kaleido/Chromium does not crash when /tmp is full.
KALEIDO_TMP = PROJECT_ROOT / "tmp" / "kaleido_tmp"
KALEIDO_TMP.mkdir(parents=True, exist_ok=True)
os.environ["TMPDIR"] = str(KALEIDO_TMP)
tempfile.tempdir = str(KALEIDO_TMP)

# Reduce reliance on shared memory and GPU in constrained Linux environments.
_extra_kaleido_args = ("--disable-dev-shm-usage", "--disable-gpu", "--single-process")
for _arg in _extra_kaleido_args:
    if _arg not in pio.kaleido.scope.chromium_args:
        pio.kaleido.scope.chromium_args += (_arg,)

In [2]:
from src.ExperimentSet import ExperimentSet

In [3]:
def get_unique_path(path):
    path = Path(path)
    if not path.exists():
        return path

    match = re.search(r'(\d+)$', path.stem)
    if match:
        base = path.stem[:match.start()]
        num = int(match.group(1)) + 1
    else:
        base = path.stem
        num = 1

    new_path = path.with_name(f"{base}{num}{path.suffix}")
    # keep incrementing until we find one that doesn't exist
    while new_path.exists():
        num += 1
        new_path = path.with_name(f"{base}{num}{path.suffix}")

    return new_path

In [4]:
es_dir = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V11'
exset = ExperimentSet(es_dir)
exset.set_retrieval_list()
exset.load_retrievals(iteration=-1, extinction_th=[0.3, np.inf])

In [5]:
isomin=1
isomax=40
opacity=0.1
surface_count=30
cmin=0
cmax=40

# rnames = ['aod0p150', 'default', 'aod0p250']
# outdir = es_dir / 'figures' / 'aerosol'
# rnames = ['periodic', 'default']
# outdir = es_dir / 'figures' / 'hbc'
# rnames = ['unconstrained', 'default']
# outdir = es_dir / 'figures' / 'cvm'
# rnames = ['surf4p0', 'default']
# outdir = es_dir / 'figures' / 'surf_wind'
# rnames = ['CTM', 'default', 'CTP']
# outdir = es_dir / 'figures' / 'cloud_motionCT'
# rnames = ['ATM', 'default', 'ATP']
# outdir = es_dir / 'figures' / 'cloud_motionAT'

rnames = ['default', 'unconstrained', 'default', 'surf4p0', 'aod0p150', 'default', 'aod0p250', 'default', 'periodic', 'ATM_unc', 'unconstrained', 'ATP_unc', 'CTM_unc', 'unconstrained', 'CTP_unc']
outdir = es_dir / 'figures' / '3Dextinction2'

rs = [exset.get_retrieval(r) for r in rnames]
(outdir / 'frames').mkdir(parents=True, exist_ok=True)

for i, r in enumerate(rs):
    cloud = r.solvers_retrieval[0.67].medium['cloud']
    cond = cloud.extinction.data >= 0.3

    X, Y, Z = np.meshgrid(cloud.x.data, cloud.y.data, cloud.z.data, indexing='ij')
    data = go.Volume(
        x=X.flatten(),
        y=Y.flatten(),
        z=Z.flatten(),
        value=cloud.extinction.data.flatten(),
        isomin=isomin,
        isomax=isomax,
        opacity=opacity,
        surface_count=surface_count,
        showscale=True,
        cmin=cmin,
        cmax=cmax,
        # colorscale='YlOrBr',
        colorscale='Blues',
        colorbar=dict(
            title='',
            titleside='right',
            tickmode='array',
            tickvals=[cmin, 0.5*(cmin+cmax), cmax],
            ticktext=['Low', 'Medium', 'High'],
            len=0.4,
            thickness=20,
            tickfont={'size': 20}
        )
    )
    layout = go.Layout(autosize=False, width=750, height=650)
    fig = go.Figure(data=data, layout=layout)
    pad = {'b':0, 'l':0, 'r':0, 't':20, 'pad':0}
    font = {'size':16}

    scene = dict(
        xaxis=dict(title='x (km)', tickvals=[0, 4.4, 8.8, 13.2, 17.6], range=[-1, 18.6]),
        yaxis=dict(title='y (km)', tickvals=[0, 4.4, 8.8, 13.2, 17.6], range=[-1, 18.6]),
        zaxis=dict(title='z (km)', tickvals=[0, 1.4], range=[-0.1, 1.5]),
        aspectmode='manual',
        aspectratio=dict(x=1,y=1,z=0.5)
        # aspectmode='data'
    )

    theta = np.radians(68)
    phi = -0.33*np.pi
    radius = 2.4
    h = radius * np.sin(theta)

    camera = dict(
        up=dict(x=0, y=0, z=1.0),
        center=dict(x=0, y=0, z=0),
        eye=dict(x=h*np.cos(phi), y=h*np.sin(phi), z=radius*np.cos(theta))
    )
    fig.update_layout(
        title=dict(
            text=str(r.exp_name),
            x=0.5,
            y=0.80,
            xanchor='center',
            yanchor='top',
            font=dict(size=16)
        ),
        scene_camera=camera,
        margin=pad,
        font=font,
        scene=scene,
        # paper_bgcolor='rgba(0,0,0,0)',
        # plot_bgcolor='rgba(1,1,1,1)'
    )

    outfile = outdir / 'frames' / f'frame{i:02d}.png'
    fig.write_image(str(outfile))


In [6]:
import glob
from figs_to_html import figs_to_html

# frames = glob.glob('/data/keeling/a/johntl3/b/LundstromPub2025/notebooks/figures/V11_3Dextinction/*.png')
frames = glob.glob(str(outdir / 'frames' / '*.png'))
figs_to_html(frames, output_path=outdir / '3Dextinction.html', titles=rnames)

PosixPath('/data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V11/figures/3Dextinction2/3Dextinction.html')